<a href="https://colab.research.google.com/github/hyunkyung31/DeepLearningProject/blob/main/hyunkyung/ARCADE_%EB%B0%8F_%EC%95%88%EC%A0%95%ED%99%94.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 1. ARCADE 압축 해제
import zipfile
from pathlib import Path

ZIP_PATH = "/content/drive/MyDrive/🐥new_2_project/04_DL_project/02_dataset/arcade.zip"   # 다운로드한 실제 파일명으로 바꾸기
EXTRACT_DIR = Path("/content/arcade_raw")
EXTRACT_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

print("압축 해제 완료:", EXTRACT_DIR)

압축 해제 완료: /content/arcade_raw


In [ ]:
# @title 2. 폴더 구조 확인
import os

for root, dirs, files in os.walk(EXTRACT_DIR):
    depth = root.replace(str(EXTRACT_DIR), "").count(os.sep)
    if depth > 2:   # 너무 깊이 안 들어가고 상위 구조만
        continue
    indent = "  " * depth
    print(f"{indent}{os.path.basename(root)}/")
    for f in files[:3]:
        print(f"{indent}  {f}")

arcade_raw/
  arcade/
    syntax/
    stenosis/


In [ ]:
# @title stenosis 폴더 안쪽 확인
import os
ARCADE_ROOT = EXTRACT_DIR / "arcade"   # /content/arcade_raw/arcade
stenosis_dir = ARCADE_ROOT / "stenosis"
for root, dirs, files in os.walk(stenosis_dir):
    depth = root.replace(str(stenosis_dir), "").count(os.sep)
    if depth > 2:
        continue
    indent = "  " * depth
    print(f"{indent}{os.path.basename(root)}/")
    for f in files[:5]:
        print(f"{indent}  {f}")

stenosis/
  train/
    images/
      694.png
      716.png
      308.png
      190.png
      586.png
    annotations/
      train.json
  test/
    images/
      190.png
      244.png
      216.png
      287.png
      150.png
    annotations/
      test.json
  val/
    images/
      190.png
      150.png
      166.png
      160.png
      44.png
    annotations/
      val.json


In [ ]:
# @title JSON 확인
import json

# 아래 두 줄 중 위 폴더 구조 확인 결과에 맞는 쪽으로 경로 조정
ann_candidates = list((stenosis_dir / "train").glob("**/*.json"))
print("찾은 json 파일들:", ann_candidates)

ann_path = ann_candidates[0]
data = json.loads(ann_path.read_text())
print("categories:", data["categories"])
print("images 개수:", len(data["images"]))
print("annotations 개수:", len(data["annotations"]))
print("샘플 image:", data["images"][0])
print("샘플 annotation:", data["annotations"][0])

찾은 json 파일들: [PosixPath('/content/arcade_raw/arcade/stenosis/train/annotations/train.json')]
categories: [{'id': 1, 'name': '1', 'supercategory': ''}, {'id': 2, 'name': '2', 'supercategory': ''}, {'id': 3, 'name': '3', 'supercategory': ''}, {'id': 4, 'name': '4', 'supercategory': ''}, {'id': 5, 'name': '5', 'supercategory': ''}, {'id': 6, 'name': '6', 'supercategory': ''}, {'id': 7, 'name': '7', 'supercategory': ''}, {'id': 8, 'name': '8', 'supercategory': ''}, {'id': 9, 'name': '9', 'supercategory': ''}, {'id': 10, 'name': '9a', 'supercategory': ''}, {'id': 11, 'name': '10', 'supercategory': ''}, {'id': 12, 'name': '10a', 'supercategory': ''}, {'id': 13, 'name': '11', 'supercategory': ''}, {'id': 14, 'name': '12', 'supercategory': ''}, {'id': 15, 'name': '12a', 'supercategory': ''}, {'id': 16, 'name': '13', 'supercategory': ''}, {'id': 17, 'name': '14', 'supercategory': ''}, {'id': 18, 'name': '14a', 'supercategory': ''}, {'id': 19, 'name': '15', 'supercategory': ''}, {'id': 20, 'name

In [ ]:
# @title convert_split 함수 정의 (먼저 실행)
import json, os
from pathlib import Path

def find_stenosis_category_ids(categories):
    return [c["id"] for c in categories if "sten" in c.get("name", "").lower()]

def convert_split(arcade_root, split, out_root, out_split="train"):
    arcade_root, out_root = Path(arcade_root), Path(out_root)
    img_dir = arcade_root / "stenosis" / split / "images"
    ann_path = arcade_root / "stenosis" / split / "annotations" / f"{split}.json"
    if not ann_path.exists():
        cands = list((arcade_root / "stenosis" / split).glob("**/*.json"))
        if not cands:
            print(f"[WARN] {split} json 못 찾음"); return 0
        ann_path = cands[0]

    data = json.loads(ann_path.read_text())
    images = {im["id"]: im for im in data["images"]}
    categories = data.get("categories", [])
    stenosis_ids = find_stenosis_category_ids(categories)
    if stenosis_ids:
        print(f"[{split}] stenosis category id(s)={stenosis_ids}")
    else:
        print(f"[{split}] [주의] 'sten' 카테고리 못 찾음 -> 전체를 lesion으로 처리. "
              f"전체 카테고리={[c['name'] for c in categories]}")

    boxes_per_image = {}
    for ann in data["annotations"]:
        if stenosis_ids and ann["category_id"] not in stenosis_ids:
            continue
        boxes_per_image.setdefault(ann["image_id"], []).append(ann["bbox"])

In [ ]:
ARCADE_ROOT = "/content/arcade_raw/arcade"

convert_split(ARCADE_ROOT, "train", "/content/arcade_yolo", "train")

[train] stenosis category id(s)=[26]


In [ ]:
# @title 변환 결과 직접 확인
from pathlib import Path

img_dir = Path("/content/arcade_yolo/images/train")
lbl_dir = Path("/content/arcade_yolo/labels/train")

n_img = len(list(img_dir.glob("*.png"))) if img_dir.exists() else 0
n_lbl = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0
print(f"이미지: {n_img}장, 라벨: {n_lbl}장")

# 라벨 파일 몇 개 내용도 확인
for f in list(lbl_dir.glob("*.txt"))[:3]:
    print(f.name, "->", f.read_text().strip())

이미지: 0장, 라벨: 0장


In [ ]:
# @title 디버그: 어느 단계에서 0이 되는지 확인
import json
from pathlib import Path

ARCADE_ROOT = Path("/content/arcade_raw/arcade")
split = "train"

img_dir = ARCADE_ROOT / "stenosis" / split / "images"
ann_path = ARCADE_ROOT / "stenosis" / split / "annotations" / f"{split}.json"

print("ann_path 존재?", ann_path.exists())
print("img_dir 존재?", img_dir.exists())

data = json.loads(ann_path.read_text())
images = {im["id"]: im for im in data["images"]}
categories = data.get("categories", [])
stenosis_ids = [c["id"] for c in categories if "sten" in c.get("name", "").lower()]
print("stenosis_ids:", stenosis_ids)

boxes_per_image = {}
for ann in data["annotations"]:
    if stenosis_ids and ann["category_id"] not in stenosis_ids:
        continue
    boxes_per_image.setdefault(ann["image_id"], []).append(ann["bbox"])

print("stenosis annotation 매칭된 image 개수:", len(boxes_per_image))

# 실제 파일이 존재하는지 3개만 확인
sample_ids = list(boxes_per_image.keys())[:3]
for image_id in sample_ids:
    im = images.get(image_id)
    print(image_id, "-> im 있음?", im is not None)
    if im:
        src_img = img_dir / im["file_name"]
        print("   src_img:", src_img, "존재?", src_img.exists())

ann_path 존재? True
img_dir 존재? True
stenosis_ids: [26]
stenosis annotation 매칭된 image 개수: 997
676 -> im 있음? True
   src_img: /content/arcade_raw/arcade/stenosis/train/images/676.png 존재? True
960 -> im 있음? True
   src_img: /content/arcade_raw/arcade/stenosis/train/images/960.png 존재? True
99 -> im 있음? True
   src_img: /content/arcade_raw/arcade/stenosis/train/images/99.png 존재? True


In [ ]:
# @title 실제 변환 다시 실행 (에러 노출 버전)
import os
import json
from pathlib import Path

ARCADE_ROOT = Path("/content/arcade_raw/arcade")
split = "train"
out_root = Path("/content/arcade_yolo")

img_dir = ARCADE_ROOT / "stenosis" / split / "images"
ann_path = ARCADE_ROOT / "stenosis" / split / "annotations" / f"{split}.json"

data = json.loads(ann_path.read_text())
images = {im["id"]: im for im in data["images"]}
categories = data.get("categories", [])
stenosis_ids = [c["id"] for c in categories if "sten" in c.get("name", "").lower()]

boxes_per_image = {}
for ann in data["annotations"]:
    if stenosis_ids and ann["category_id"] not in stenosis_ids:
        continue
    boxes_per_image.setdefault(ann["image_id"], []).append(ann["bbox"])

out_img_dir = out_root / "images" / split
out_lbl_dir = out_root / "labels" / split
out_img_dir.mkdir(parents=True, exist_ok=True)
out_lbl_dir.mkdir(parents=True, exist_ok=True)

n = 0
errors = []
for image_id, boxes in boxes_per_image.items():
    try:
        im = images.get(image_id)
        if im is None:
            continue
        img_w, img_h = im["width"], im["height"]
        src_img = img_dir / im["file_name"]
        if not src_img.exists():
            continue

        stem = f"arcade_{split}_{image_id}"
        dst_img = out_img_dir / f"{stem}{src_img.suffix}"
        if not dst_img.exists():
            os.symlink(src_img.resolve(), dst_img)

        lines = []
        for x, y, w, h in boxes:
            if w <= 0 or h <= 0:
                continue
            cx, cy = (x + w/2) / img_w, (y + h/2) / img_h
            lines.append(f"0 {cx:.6f} {cy:.6f} {w/img_w:.6f} {h/img_h:.6f}")
        (out_lbl_dir / f"{stem}.txt").write_text("\n".join(lines))
        n += 1
    except Exception as e:
        errors.append((image_id, repr(e)))

print("변환 완료:", n)
print("에러 개수:", len(errors))
if errors:
    print("첫 5개 에러:", errors[:5])

# 실제로 폴더에 뭐가 생겼는지 바로 확인
print("images/train 파일 수:", len(list(out_img_dir.glob("*"))))
print("labels/train 파일 수:", len(list(out_lbl_dir.glob("*"))))

변환 완료: 997
에러 개수: 0
images/train 파일 수: 997
labels/train 파일 수: 997


In [ ]:
# @title 샘플 라벨 확인
sample = list(out_lbl_dir.glob("*.txt"))[0]
print(sample.name)
print(sample.read_text())

arcade_train_504.txt
0 0.400264 0.456055 0.121816 0.105469
0 0.396738 0.333496 0.122070 0.060039


In [ ]:
# @title CADICA 데이터셋 존재 확인
from pathlib import Path

p = Path("/content/cadica_yolo")
print("존재?", p.exists())
if p.exists():
    for split in ("train", "val", "test"):
        imgs = list((p/"images"/split).glob("*")) if (p/"images"/split).exists() else []
        print(f"{split}: {len(imgs)}장")

존재? False


In [ ]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/🐥new_2_project/04_DL_project/02_dataset/CADICA.zip'
extract_path = '/content/cadica_dataset/'

if os.path.exists(zip_path) :
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(f'압축 해제 완료, 저장위치 : {extract_path}')
else :
    print(f'에러 {zip_path}에 파일이 존재하지 않습니다')

압축 해제 완료, 저장위치 : /content/cadica_dataset/


In [ ]:
# @title 1. Drive 마운트 + 경로 확인

from pathlib import Path
CADICA_RAW = Path("/content/cadica_dataset/CADICA/selectedVideos")
print("존재?", CADICA_RAW.exists())
print(list(CADICA_RAW.glob("p*"))[:5])  # p1, p2, ... 폴더들 보이면 정상

존재? True
[PosixPath('/content/cadica_dataset/CADICA/selectedVideos/p19'), PosixPath('/content/cadica_dataset/CADICA/selectedVideos/p5'), PosixPath('/content/cadica_dataset/CADICA/selectedVideos/p34'), PosixPath('/content/cadica_dataset/CADICA/selectedVideos/p25'), PosixPath('/content/cadica_dataset/CADICA/selectedVideos/p7')]


In [ ]:
# @title 2. CADICA -> YOLO 재생성
import os, re
from pathlib import Path
import pandas as pd
from PIL import Image

SPLIT_CSV = Path("/content/drive/MyDrive/🐥new_2_project/04_DL_project/08_전처리/common_split.csv")
OUT_ROOT = Path("/content/cadica_yolo")

split_df = pd.read_csv(SPLIT_CSV)

def find_frame_image(video_dir: Path, gt_stem: str):
    input_dir = video_dir / "input"
    cand = input_dir / f"{gt_stem}.png"
    if cand.exists():
        return cand
    m = re.search(r"(\d+)$", gt_stem)
    if m:
        cand2 = input_dir / f"{m.group(1)}.png"
        if cand2.exists():
            return cand2
    return None

n_written = {"train": 0, "val": 0, "test": 0}
n_skipped_no_gt = 0
n_skipped_no_img = 0

for row in split_df.itertuples():
    if not row.has_groundtruth:
        continue
    video_dir = CADICA_RAW / row.patient_id / row.video_id
    gt_dir = video_dir / "groundtruth"
    if not gt_dir.exists():
        n_skipped_no_gt += 1
        continue

    split = row.split
    out_img_dir = OUT_ROOT / "images" / split
    out_lbl_dir = OUT_ROOT / "labels" / split
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    for gt_file in sorted(gt_dir.glob("*.txt")):
        if "groundTruthTable" in gt_file.name:
            continue
        img_path = find_frame_image(video_dir, gt_file.stem)
        if img_path is None:
            n_skipped_no_img += 1
            continue

        with Image.open(img_path) as im:
            img_w, img_h = im.size

        lines = []
        for line in gt_file.read_text().splitlines():
            parts = line.split()
            if len(parts) < 4:
                continue
            x, y, w, h = map(float, parts[:4])
            if w <= 0 or h <= 0:
                continue
            cx, cy = (x + w/2)/img_w, (y + h/2)/img_h
            lines.append(f"0 {cx:.6f} {cy:.6f} {w/img_w:.6f} {h/img_h:.6f}")

        stem = gt_file.stem
        dst_img = out_img_dir / f"{stem}{img_path.suffix}"
        if not dst_img.exists():
            os.symlink(img_path.resolve(), dst_img)
        (out_lbl_dir / f"{stem}.txt").write_text("\n".join(lines))
        n_written[split] += 1

print("변환 완료:", n_written, " 합계:", sum(n_written.values()))
print("groundtruth 폴더 없음(video):", n_skipped_no_gt)
print("이미지 못 찾음(frame):", n_skipped_no_img)

(OUT_ROOT / "data.yaml").write_text(
    f"path: {OUT_ROOT}\ntrain: images/train\nval: images/val\ntest: images/test\nnc: 1\nnames: ['lesion']\n"
)

변환 완료: {'train': 3077, 'val': 335, 'test': 273}  합계: 3685
groundtruth 폴더 없음(video): 0
이미지 못 찾음(frame): 0


105

In [ ]:
# @title CADICA + ARCADE 병합
import os
from pathlib import Path

def link_dir(src_dir, dst_dir):
    src_dir, dst_dir = Path(src_dir), Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    if not src_dir.exists():
        return 0
    n = 0
    for f in src_dir.iterdir():
        if not f.is_file():
            continue
        dst = dst_dir / f.name
        if not dst.exists():
            os.symlink(f.resolve(), dst)
        n += 1
    return n

CADICA = "/content/cadica_yolo"
ARCADE = "/content/arcade_yolo"
OUT = Path("/content/cadica_arcade_yolo")

counts = {}
for split in ("val", "test"):  # CADICA만
    n_img = link_dir(f"{CADICA}/images/{split}", OUT/"images"/split)
    link_dir(f"{CADICA}/labels/{split}", OUT/"labels"/split)
    counts[split] = {"CADICA": n_img}

n_c = link_dir(f"{CADICA}/images/train", OUT/"images"/"train")
link_dir(f"{CADICA}/labels/train", OUT/"labels"/"train")
n_a = link_dir(f"{ARCADE}/images/train", OUT/"images"/"train")
link_dir(f"{ARCADE}/labels/train", OUT/"labels"/"train")
counts["train"] = {"CADICA": n_c, "ARCADE": n_a}

(OUT/"data.yaml").write_text(
    f"path: {OUT}\ntrain: images/train\nval: images/val\ntest: images/test\nnc: 1\nnames: ['lesion']\n"
)
print("=== 병합 결과 ===")
for split, d in counts.items():
    print(f"{split:6} " + ", ".join(f"{k}={v}" for k, v in d.items()) + f"  total={sum(d.values())}")

=== 병합 결과 ===
val    CADICA=335  total=335
test   CADICA=273  total=273
train  CADICA=3077, ARCADE=997  total=4074


In [ ]:
# @title Ultralytics 설치
!pip install -q ultralytics


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.5 MB/s eta 0:00:00


In [ ]:
from pathlib import Path

from ultralytics import RTDETR

BASE = Path(
    "/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result"
)
DATA_YAML = "/content/cadica_arcade_yolo/data.yaml"
RUN_NAME = "train_cadica_arcade_v1"

COMMON_TRAIN_KWARGS = dict(
    epochs=50,
    imgsz=640,
    batch=8,
    lr0=0.0005,
    optimizer="AdamW",
    amp=True,
    patience=20,
    project=str(BASE),
)

# CADICA-only baseline (train_v3_amp_on) — 비교용 하드코딩
BASELINE = {
    "val": dict(P=0.229, R=0.219, mAP50=0.117, mAP50_95=0.030),
    "test": dict(P=0.592, R=0.289, mAP50=0.308, mAP50_95=0.150),
}


def train():
    model = RTDETR("rtdetr-l.pt")  # 원본 pretrained에서 새로 시작
    model.train(data=DATA_YAML, name=RUN_NAME, **COMMON_TRAIN_KWARGS)
    return model


def eval_and_compare(imgsz: int = 640):
    weights = BASE / RUN_NAME / "weights" / "best.pt"
    if not weights.exists():
        raise FileNotFoundError(f"{weights} 없음 — train() 먼저 실행")

    model = RTDETR(str(weights))
    val = model.val(data=DATA_YAML, split="val", imgsz=imgsz)
    test = model.val(data=DATA_YAML, split="test", imgsz=imgsz)

    result = {
        "val": dict(P=val.box.mp, R=val.box.mr, mAP50=val.box.map50, mAP50_95=val.box.map),
        "test": dict(P=test.box.mp, R=test.box.mr, mAP50=test.box.map50, mAP50_95=test.box.map),
    }

    header = f"{'split':6} {'group':16} {'P':>6} {'R':>6} {'mAP50':>7} {'mAP50-95':>9}"
    print(header)
    print("-" * len(header))
    for split in ("val", "test"):
        b = BASELINE[split]
        print(f"{split:6} {'CADICA-only(합)':16} {b['P']:6.3f} {b['R']:6.3f} "
              f"{b['mAP50']:7.3f} {b['mAP50_95']:9.3f}")
        r = result[split]
        print(f"{split:6} {'CADICA+ARCADE':16} {r['P']:6.3f} {r['R']:6.3f} "
              f"{r['mAP50']:7.3f} {r['mAP50_95']:9.3f}")

    return result


if __name__ == "__main__":
    train()
    eval_and_compare()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/cadica_arcade_yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, 

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/50      6.66G       1.33     0.6815     0.4613          5        640: 100% ━━━━━━━━━━━━ 510/510 1.5it/s 5:51
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 1.7it/s 12.6s
                   all        335        506     0.0156      0.241     0.0122    0.00215

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/50      7.05G     0.9411     0.8842     0.3693         13        640: 0% ──────────── 0/510  0.7s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/50      7.05G     0.8743     0.8551     0.2446         12        640: 100% ━━━━━━━━━━━━ 510/510 1.5it/s 5:32
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.9it/s 7.2s
                   all        335        506     0.0197      0.192     0.0144    0.00375

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/50      7.05G     0.8634      0.869     0.2423         19        640: 0% ──────────── 0/510  0.6s

...

      28/50      7.05G     0.5091     0.5342     0.1366          6        640: 100% ━━━━━━━━━━━━ 510/510 1.5it/s 5:29
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 3.1it/s 6.8s
                   all        335        506      0.217      0.198     0.0877     0.0243

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      29/50      7.05G     0.5711     0.5692     0.1609         11        640: 0% ──────────── 0/510  0.8s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      29/50      7.05G     0.5031     0.5267     0.1339          0        640: 100% ━━━━━━━━━━━━ 510/510 1.5it/s 5:33
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 3.0it/s 7.1s
                   all        335        506      0.206      0.191      0.103      0.032

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      30/50      7.05G     0.5229     0.4362     0.1369         11        640: 0% ──────────── 0/510  0.6s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      30/50      7.05G     0.4862     0.5065     0.1237         11        640: 21% ━━╸───────── 109/510 1.6it/s 1:14<4:05

In [ ]:
from ultralytics import RTDETR

last = "/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_cadica_arcade_v1/weights/last.pt"
model = RTDETR(last)
model.train(resume=True)

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/cadica_arcade_yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_cadica_arcade_v1/weights/last.pt, momentum=0.937, mosaic=1.0, mul

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      30/50      7.31G     0.4953     0.5108      0.132          5        640: 100% ━━━━━━━━━━━━ 510/510 1.5it/s 5:47
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 1.7it/s 12.7s
                   all        335        506      0.186      0.229     0.0984     0.0291

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      31/50      7.29G     0.4618     0.4965      0.148         13        640: 0% ──────────── 0/510  0.7s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      31/50       7.3G     0.4898     0.5101     0.1308         12        640: 100% ━━━━━━━━━━━━ 510/510 1.5it/s 5:39
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.8it/s 7.6s
                   all        335        506      0.175       0.15     0.0574     0.0156

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      32/50      7.35G     0.4788     0.4966     0.1065         19        640: 0% ──────────── 0/510  0.6s

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      49/50       7.4G     0.3386     0.3869    0.09846          3        640: 100% ━━━━━━━━━━━━ 510/510 1.5it/s 5:38
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.8it/s 7.5s
                   all        335        506      0.216       0.16      0.076     0.0182

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      50/50      7.37G     0.2178     0.3412     0.0799         12        640: 0% ──────────── 0/510  0.7s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      50/50      7.38G     0.3319     0.3822    0.09853          3        640: 100% ━━━━━━━━━━━━ 510/510 1.5it/s 5:37
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.8it/s 7.4s
                   all        335        506      0.236       0.14     0.0675     0.0161
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 30, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

21 epochs completed in 2.035 hours.
Optimizer stripped from /content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_cadica_arcade_v1/weights/last.pt, 66.3MB
Optimizer stripped from /content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_cadica_arcade_v1/weights/best.pt, 66.3MB

Validating /content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_res

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78d3a0ad23f0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
from ultralytics import RTDETR
BASE = "/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result"
w = f"{BASE}/train_cadica_arcade_v1/weights/best.pt"
m = RTDETR(w)
yaml = "/content/cadica_arcade_yolo/data.yaml"
print("VAL ", m.val(data=yaml, split="val", imgsz=640))
print("TEST", m.val(data=yaml, split="test", imgsz=640))

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2115.7±589.8 MB/s, size: 96.2 KB)
val: Scanning /content/cadica_arcade_yolo/labels/val.cache... 335 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 335/335 93.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 1.2it/s 17.5s
                   all        335        506      0.273      0.245      0.138     0.0404
Speed: 2.5ms preprocess, 45.4ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /content/runs/detect/val
VAL  ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78d402d24560>
curves: ['Precision-Recall(B)', 'F1-Confiden

In [ ]:
# @title 빡센 안정화 해보기

"""
RT-DETR CADICA-only 강한 안정화 학습 + 평가 (Colab)

데이터/split은 baseline과 동일 (/content/cadica_yolo).
바꾸는 것은 학습 스케줄·augmentation 강도뿐.
"""

from pathlib import Path

from ultralytics import RTDETR

BASE = Path(
    "/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result"
)
DATA_YAML = "/content/cadica_yolo/data.yaml"
RUN_NAME = "train_stability_v1"

# CADICA-only baseline (train_v3_amp_on)
BASELINE = {
    "val": dict(P=0.229, R=0.219, mAP50=0.117, mAP50_95=0.030),
    "test": dict(P=0.592, R=0.289, mAP50=0.308, mAP50_95=0.150),
}

STABILITY_KWARGS = dict(
    data=DATA_YAML,
    name=RUN_NAME,
    project=str(BASE),
    epochs=100,
    imgsz=640,
    batch=8,
    optimizer="AdamW",
    amp=True,
    # --- stronger stability vs baseline ---
    lr0=0.00015,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=6.0,
    weight_decay=0.0001,
    dropout=0.1,  # extra regularization; RT-DETR decoder supports this
    patience=35,
    mosaic=0.5,
    close_mosaic=20,
    mixup=0.0,
    copy_paste=0.0,
    degrees=0.0,
    shear=0.0,
    perspective=0.0,
    scale=0.3,
    translate=0.05,
    fliplr=0.5,
    flipud=0.0,
    hsv_h=0.01,
    hsv_s=0.4,
    hsv_v=0.3,
)

# Stage-2: cold fine-tune from stage-1 best.pt, near-zero aug, very low lr.
# Only run this if train()+eval_and_compare() shows stage-1 beats baseline
# on test, or is close but still oscillating on val.
FINETUNE_KWARGS = dict(
    data=DATA_YAML,
    name="train_stability_v1_ft",
    project=str(BASE),
    epochs=40,
    imgsz=640,
    batch=8,
    optimizer="AdamW",
    amp=True,
    lr0=0.00003,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=0,
    weight_decay=0.0001,
    dropout=0.1,
    patience=15,
    mosaic=0.0,
    close_mosaic=0,
    mixup=0.0,
    copy_paste=0.0,
    degrees=0.0,
    shear=0.0,
    perspective=0.0,
    scale=0.1,
    translate=0.02,
    fliplr=0.5,
    flipud=0.0,
    hsv_h=0.005,
    hsv_s=0.2,
    hsv_v=0.15,
)


def train():
    model = RTDETR("rtdetr-l.pt")
    model.train(**STABILITY_KWARGS)
    return model


def finetune_stage2():
    """Cold fine-tune stage-1 best.pt with near-zero aug + tiny lr (S3)."""
    stage1_best = BASE / RUN_NAME / "weights" / "best.pt"
    if not stage1_best.exists():
        raise FileNotFoundError(f"{stage1_best} 없음 — train() 먼저 실행")
    model = RTDETR(str(stage1_best))
    model.train(**FINETUNE_KWARGS)
    return model


def eval_and_compare(run_name: str = RUN_NAME, label: str = "stability_v1", imgsz: int = 640):
    weights = BASE / run_name / "weights" / "best.pt"
    if not weights.exists():
        raise FileNotFoundError(f"{weights} 없음 — train() 먼저 실행")

    model = RTDETR(str(weights))
    val = model.val(data=DATA_YAML, split="val", imgsz=imgsz)
    test = model.val(data=DATA_YAML, split="test", imgsz=imgsz)

    result = {
        "val": dict(
            P=val.box.mp, R=val.box.mr, mAP50=val.box.map50, mAP50_95=val.box.map
        ),
        "test": dict(
            P=test.box.mp, R=test.box.mr, mAP50=test.box.map50, mAP50_95=test.box.map
        ),
    }

    header = f"{'split':6} {'group':18} {'P':>6} {'R':>6} {'mAP50':>7} {'mAP50-95':>9}"
    print(header)
    print("-" * len(header))
    for split in ("val", "test"):
        b = BASELINE[split]
        print(
            f"{split:6} {'baseline(v3)':18} {b['P']:6.3f} {b['R']:6.3f} "
            f"{b['mAP50']:7.3f} {b['mAP50_95']:9.3f}"
        )
        r = result[split]
        print(
            f"{split:6} {label:18} {r['P']:6.3f} {r['R']:6.3f} "
            f"{r['mAP50']:7.3f} {r['mAP50_95']:9.3f}"
        )
    return result


if __name__ == "__main__":
    train()
    eval_and_compare(RUN_NAME, "stability_v1")
    # If stage-1 looks promising but still noisy, uncomment:
    # finetune_stage2()
    # eval_and_compare("train_stability_v1_ft", "stability_v1_ft")

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=20, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/cadica_yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.01, hsv_s=0.4, hsv_v=0.3, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00015, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=train_stability_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/100      7.93G      1.338     0.8433     0.4766          6        640: 100% ━━━━━━━━━━━━ 385/385 1.4it/s 4:29
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.9it/s 7.3s
                   all        335        506    0.00126      0.251    0.00156   0.000258

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      2/100      7.55G      1.248     0.6023     0.3935         22        640: 0% ──────────── 0/385  0.6s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/100      7.55G     0.8649     0.9959     0.2368          6        640: 100% ━━━━━━━━━━━━ 385/385 1.5it/s 4:19
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.9it/s 7.1s
                   all        335        506     0.0191      0.168     0.0133    0.00229

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      3/100      7.48G      1.026     0.9498     0.3061         18        640: 0% ──────────── 0/385  1.2s

...

     41/100      7.53G     0.3806     0.4006    0.09498         11        640: 100% ━━━━━━━━━━━━ 385/385 1.5it/s 4:17
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.8it/s 7.5s
                   all        335        506      0.263      0.138     0.0803     0.0213

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     42/100      7.39G     0.5798     0.3918     0.1628         16        640: 0% ──────────── 0/385  0.8s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     42/100      7.39G     0.3825     0.4052    0.09643         10        640: 100% ━━━━━━━━━━━━ 385/385 1.5it/s 4:16
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.8it/s 7.5s
                   all        335        506      0.213      0.164     0.0944     0.0262

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     43/100       7.5G     0.4329     0.3904     0.1049         15        640: 0% ──────────── 0/385  0.6s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     43/100       7.5G     0.3748     0.3988     0.0939          7        640: 100% ━━━━━━━━━━━━ 385/385 1.5it/s 4:15
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.9it/s 7.2s
                   all        335        506      0.266      0.172      0.109     0.0296
EarlyStopping: Training stopped early as no improvement observed in last 35 epochs. Best results observed at epoch 8, best model saved as best.pt.
To update EarlyStopping(patience=35) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

43 epochs completed in 3.242 hours.
Optimizer stripped from /content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_stability_v1/weights/last.pt, 66.3MB
Optimizer stripped from /content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_stability_v1/weights/best.pt, 66.3MB

Validating /content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train

In [ ]:
train()
eval_and_compare(RUN_NAME, "stability_v1")

NameError: name 'train' is not defined

In [ ]:
finetune_stage2()
eval_and_compare("train_stability_v1_ft", "stability_v1_ft")